# 데이터 전처리
- `데이터 전처리`란 데이터를 모델이나 분석에 쓰기 좋은 상태로 바꾸는 판단 과정
- 데이터를 확인하고, 중복값 제거, 결측치 처리, 변수 변환, 이상값 처리, 정규화 등의 작업을 진행함

## 00. 학습 흐름
1. 데이터 확인: 행/열, 자료형, 결측치, 중복 확인
2. 기본 정제: 중복값 제거, 결측치 확인/처리
3. 전처리 보강: 변수 변환, 값 변환, 이상값 처리, 정규화, 참고 개념


In [2]:
import pandas as pd

In [3]:
contact_df = pd.read_csv('data/contacts.csv')
contact_df.head()

,Name,Phone,Email
0,김민수,010-1234-5678,minsu.kim@gmail.com
1,이지은,010-2345-6789,jieun.lee@naver.com
2,박철수,010-3456-7890,chulsoo.park@hotmail.com
3,홍길동,010-4567-8901,gildong.hong@daum.net
4,김영희,010-5678-9012,younghee.kim@gmail.com


In [4]:
contact_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Name    77 non-null     str  
 1   Phone   71 non-null     str  
 2   Email   76 non-null     str  
dtypes: str(3)
memory usage: 5.1 KB


In [5]:
# 수치형데이터가 없는 경우, 최빈값 확인
contact_df.describe()

,Name,Phone,Email
count,77,71,76
unique,75,63,75
top,박철수,010-2345-6789,dahyun.jung@naver.com
freq,2,2,2


In [6]:
# 중복값 확인
print(contact_df[contact_df['Name'] == '박철수'])
print(contact_df.loc[contact_df['Name'] == '박철수'])

print(contact_df[contact_df['Phone'] == '010-2345-6789'])
print(contact_df[contact_df['Email'] == 'dahyun.jung@naver.com'])

   Name          Phone                     Email
2   박철수  010-3456-7890  chulsoo.park@hotmail.com
31  박철수  010-1234-5681    chulsoo.park@gmail.com
   Name          Phone                     Email
2   박철수  010-3456-7890  chulsoo.park@hotmail.com
31  박철수  010-1234-5681    chulsoo.park@gmail.com
   Name          Phone                     Email
1   이지은  010-2345-6789       jieun.lee@naver.com
71  엄정희  010-2345-6789  jeonghee.eom@hotmail.com
   Name          Phone                  Email
32  정다현  010-2345-6782  dahyun.jung@naver.com
67  정다현  010-2345-6782  dahyun.jung@naver.com


## 01. 중복값 처리


In [7]:
# 중복여부
display(contact_df[contact_df.duplicated()]) # 모든 컬럼값이 동일한 경우 중복으로 간주

print("전체 중복 행 수:", contact_df.duplicated().sum())
print("이름 중복 수:", contact_df.duplicated("Name").sum())
print("이름+전화번호 중복 수:", contact_df.duplicated(["Name", "Phone"]).sum())

print("이름+전화번호 기준 중복 후보:")
display(contact_df[contact_df.duplicated(["Name", "Phone"], keep=False)])

contact_df[contact_df.duplicated()].index # 인덱스 조회

,Name,Phone,Email
67,정다현,010-2345-6782,dahyun.jung@naver.com


전체 중복 행 수: 1
이름 중복 수: 2
이름+전화번호 중복 수: 1
이름+전화번호 기준 중복 후보:


,Name,Phone,Email
32,정다현,010-2345-6782,dahyun.jung@naver.com
67,정다현,010-2345-6782,dahyun.jung@naver.com


RangeIndex(start=67, stop=68, step=1)

In [8]:
# 중복 제거 기준
# - 이름만 같은 경우는 동명이인 가능성이 있어 바로 삭제하지 않습니다.
# - 이름과 전화번호가 함께 같으면 중복 연락처일 가능성이 높습니다.
contact_df = contact_df.drop_duplicates(["Name", "Phone"], keep='first')
print("중복 제거 후 행/열:", contact_df.shape)
display(contact_df.head())

중복 제거 후 행/열: (76, 3)


,Name,Phone,Email
0,김민수,010-1234-5678,minsu.kim@gmail.com
1,이지은,010-2345-6789,jieun.lee@naver.com
2,박철수,010-3456-7890,chulsoo.park@hotmail.com
3,홍길동,010-4567-8901,gildong.hong@daum.net
4,김영희,010-5678-9012,younghee.kim@gmail.com


## 02. 결측치 처리
- `info()`: 컬럼별 데이터 개수와 자료형을 확인해 결측치 존재 여부를 빠르게 파악함
- `isna()`: 각 데이터가 결측치인지 `True/False`로 확인함
- `isnull()`: `isna()`와 같은 기능으로, 각 데이터가 결측치인지 확인함
- `fillna()`: 결측치를 특정 값, 평균값, 최빈값 등으로 채움

In [9]:
# 결측치 확인
contact_df.isna() # 모든컬럼

,Name,Phone,Email
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
72,False,False,False
73,False,False,False
74,False,False,False
75,False,False,False


In [10]:
contact_df.isna().sum()
contact_df.isnull().sum()

Name     0
Phone    6
Email    1
dtype: int64

In [11]:
# 특정컬럼만 결측치 확인
contact_df['Phone'].isna().sum()

np.int64(6)

In [12]:
# 결측치처리
# 1. 기본값 처리: 0
# 2. (수치형) 평균값 처리
# 3. 결측치가 있는 행 제거
contact_df[contact_df['Phone'].isna()]
contact_df['Phone'] = contact_df['Phone'].fillna('010-0000-0000')
contact_df

,Name,Phone,Email
0,김민수,010-1234-5678,minsu.kim@gmail.com
1,이지은,010-2345-6789,jieun.lee@naver.com
2,박철수,010-3456-7890,chulsoo.park@hotmail.com
3,홍길동,010-4567-8901,gildong.hong@daum.net
4,김영희,010-5678-9012,younghee.kim@gmail.com
...,...,...,...
72,범수정,010-3456-7892,soojeong.beom@gmail.com
73,이호진,010-4567-8904,hojin.lee@daum.net
74,정지윤,010-5678-9015,jungzee@naver.com
75,김지현,010-6789-0126,jh.kim@gmail.com


In [13]:
# 결측치가 있는 행 제거
contact_df = contact_df.dropna(how='any') # all 모든컬럼이 결측치인 경우, any 하나라도 결측치가 있으면 제거
contact_df.isna().sum()

Name     0
Phone    0
Email    0
dtype: int64

### 02-1. 결측치 조회 관련

1. **`any()`**:
   - 하나라도 `True`인 값이 있으면 `True`를 반환.
   - 축(axis)을 지정하면 해당 축 방향으로 `any` 조건을 평가.

   ```python
   df.isnull().any(axis=1)  # 각 행에 대해 결측값이 있는지 확인
   ```

2. **`all()`**:
   - 모든 값이 `True`이면 `True`를 반환.
   - 축(axis)을 지정하면 해당 축 방향으로 `all` 조건을 평가.

   ```python
   df.isnull().all(axis=0)  # 각 열에 대해 모두 결측값인지 확인
   ```

3. **`sum()`**:
   - `True`를 `1`, `False`를 `0`으로 간주해 합계를 계산.
   - 결측값의 개수를 빠르게 확인하는 데 사용.

   ```python
   df.isnull().sum()  # 각 열에 대한 결측값의 개수
   ```

4. **`mean()`**:
   - `True`를 `1`, `False`를 `0`으로 간주해 평균을 계산.
   - 결측값 비율을 확인할 때 유용.

   ```python
   df.isnull().mean()  # 각 열에 대한 결측값 비율
   ```

5. **`idxmax()`**:
   - 가장 먼저 `True`가 나오는 인덱스를 반환.
   - 결측값이 처음으로 나타나는 위치를 찾을 때 사용.

   ```python
   df.isnull().idxmax()  # 각 열에서 결측값이 처음 나타나는 행의 인덱스
   ```

6. **`idxmin()`**:
   - 가장 먼저 `False`가 나오는 인덱스를 반환.

   ```python
   df.isnull().idxmin()  # 결측값이 아닌 값이 처음 나타나는 인덱스
   ```

7. **`astype(int)`**:
   - `True/False`를 `1/0`으로 변환해 DataFrame의 연산에 사용.

   ```python
   df.isnull().astype(int)  # 결측값을 1로 변환
   ```
**축(`axis`) 옵션**
- **`axis=0`** (기본값): 열(column) 단위로 연산.
- **`axis=1`**: 행(row) 단위로 연산.

In [14]:
import numpy as np

data = {
    "A": [1, np.nan, 2, np.nan],
    "B": [3, 4, 5, 6]
}
df = pd.DataFrame(data)
df

,A,B
0,1.0,3
1,NaN,4
2,2.0,5
3,NaN,6


In [15]:
# any(): 결측치가 하나라도 있는 경우 True 반환
df.isna().any()

A     True
B    False
dtype: bool

In [16]:
# all(): 모두 결측치인 경우 True 반환
df.isna().all()

A    False
B    False
dtype: bool

In [17]:
# axis속성
df.isna().any(axis=0) # 행고정. 열별 결측치여부

A     True
B    False
dtype: bool

In [18]:
df.isna().any(axis=1) # 열고정. 행별 결측치여부

0    False
1     True
2    False
3     True
dtype: bool

In [19]:
# idxmin(): 결측치가 아닌 첫 데이터의 인덱스
df.isna().idxmin()

A    0
B    0
dtype: int64

In [20]:
# idxmax(): 첫 결측치 데이터의 인덱스
df.isna().idxmax()

A    1
B    0
dtype: int64

## 03. 데이터 전처리 보강

전처리는 단순히 코드를 외우는 단원이 아니라, **데이터를 모델이나 분석에 쓰기 좋은 상태로 바꾸는 판단 과정**이다.

1. 데이터 확인: 행/열, 자료형, 결측치, 중복 확인
2. 값 정리: 컬럼명, 자료형, 잘못된 값, 표현 방식 통일
3. 변수 변환: 기존 컬럼을 이용해 분석에 더 유용한 컬럼 생성
4. 결측치/이상값 처리: 삭제할지, 대체할지, 별도 표시할지 판단
5. 스케일 조정: 정규화/표준화로 값의 범위를 맞춤


In [ ]:
import pandas as pd
import numpy as np

# np.nan은 NumPy에서 제공하는 결측값 표현이며, Pandas에서는 결측치로 인식됨
# 실제 현업 데이터에서도 구매금액, 소득, 나이, 등급처럼 일부 값이 비어 있는 경우가 자주 있음
customer_df = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106, 107],
    "name": ["김민수", "이지은", "박철수", "홍길동", "김영희", "정다현", "최서준"],
    "purchase1": [120000, 350000, 80000, 500000, np.nan, 150000, 70000],
    "purchase2": [90000, 120000, 40000, 620000, 30000, np.nan, 65000],
    "income": [3200, 5400, np.nan, 9100, 2800, 4500, 3000],
    "age": [25, 38, np.nan, 45, 29, 33, np.nan],
    "grade": ["basic", "vip", "basic", "vvip", None, "vip", "basic"],
    "join_date": ["2024-01-15", "2023-11-02", "2024-03-20", "2022-07-08", "2024-04-01", "2023-05-10", "2024-02-18"],
})

display(customer_df)
customer_df.info()


### 03-1. 변수 변환: 파생변수, 요약변수, 기준점 정의
- 파생변수: 기존 컬럼을 계산해서 새로 만든 컬럼
    - 예를 들어 `purchase1`, `purchase2`만 있으면 각 구매 기록은 볼 수 있지만, 고객별 전체 구매 규모는 바로 보이지 않음.
    - 이때 `total_purchase`라는 파생 변수를 만들면 고객별 구매 규모를 한 번에 비교할 수 있음.

- 요약변수: 여러 컬럼을 하나로 요약해서 새로운 컬럼을 만든 것
- 기준점 정의: 평균, 중앙값, 특정 기준값을 기준으로 데이터를 해석하기 쉽게 바꾸는 작업


In [ ]:
# 원본 customer_df를 직접 수정하지 않기 위해 copy()로 실습용 복사본 생성
customer_work_df = customer_df.copy()

# 파생변수: 기존 컬럼을 계산해서 새 컬럼을 만드는 작업
# purchase1과 purchase2를 더해 고객별 전체 구매금액(total_purchase) 파생 변수를 만듦.
# 단, 둘 중 하나라도 결측치이면 단순 덧셈 결과도 결측치가된다.
customer_work_df["total_purchase"] = customer_work_df["purchase1"] + customer_work_df["purchase2"]

# 파생변수: 평균 구매 금액을 만듭니다.
# axis=1은 "행 방향 계산"입니다. 즉 한 고객의 purchase1, purchase2를 가로로 묶어 평균을 계산합니다.
# skipna=True는 결측치를 제외하고 계산하라는 의미이며, mean()의 기본값입니다.
customer_work_df["avg_purchase"] = customer_work_df[["purchase1", "purchase2"]].mean(axis=1, skipna=True)

# 기준점 정의: 전체 소득 평균을 기준으로 각 고객의 소득이 평균보다 얼마나 높은지/낮은지 계산
# mean()은 기본적으로 결측치를 제외하고 평균을 계산
income_mean = customer_work_df["income"].mean()
customer_work_df["income_diff"] = customer_work_df["income"] - income_mean

# 소득값 하나를 받아서 설명용 등급 문자열로 바꿔주는 함수
# apply()와 함께 사용할 예정이므로, 입력값 income은 income 컬럼의 원소 하나라고 이해하면 된다.
def categorize_income(income):
    # 결측치 비교는 income == np.nan처럼 하지 않고 pd.isna()를 사용
    if pd.isna(income):
        return "소득 정보 없음"
    if income >= income_mean + 1000:
        return "평균보다 높음"
    if income <= income_mean - 1000:
        return "평균보다 낮음"
    return "평균 근처"

# apply(): Series의 각 값을 함수에 하나씩 넣고, 반환값을 모아 새 Series를 만듦
# 여기서는 income 값 하나하나를 categorize_income()에 넣어 income_level 컬럼을 만듦
customer_work_df["income_level"] = customer_work_df["income"].apply(categorize_income)

# 결과 확인
display(customer_work_df[[
    "customer_id", "purchase1", "purchase2", "total_purchase",
    "avg_purchase", "income", "income_diff", "income_level"
]])

display(customer_work_df)
print("소득 평균:", round(income_mean, 1))


### 03-2. 값 변환: `rename`, `map`, `replace`, `astype`, `to_datetime`
전처리에서는 값 자체를 바꾸는 일도 많음

- `rename`: 컬럼명을 더 이해하기 쉽게 변경
- `map`: 값 하나하나를 다른 값으로 매핑
- `replace`: 특정 값을 다른 값으로 치환
- `astype`: 자료형 변환
- `to_datetime`: 문자열 날짜를 날짜형으로 변환


In [ ]:
# grade_map: 등급 문자열을 숫자 점수로 바꾸기 위한 매핑표
# map()을 설명할 때 "왼쪽 값이 들어오면 오른쪽 값으로 바뀜"
grade_map = {
    "basic": 1,
    "vip": 2,
    "vvip": 3,
}

# rename(): 컬럼명을 변경함
# 원본 customer_work_df의 컬럼명은 유지하고, 바뀐 컬럼명을 가진 새 DataFrame을 renamed_df를 반환 받음
# 뒤에 copy()를 붙여 이후 실습에서 원본과 독립적으로 다루기 쉽게 합니다.
renamed_df = customer_work_df.rename(columns={
    "customer_id": "id",
    "total_purchase": "total_amount",
    "avg_purchase": "avg_amount",
}).copy()

# display(customer_work_df)
# display(renamed_df)


# map(): 값 변환표를 이용해 Series의 값을 다른 값으로 변환.
# grade가 basic이면 1, vip이면 2, vvip이면 3으로 바뀝니다.
# 매핑표에 없는 값이나 결측치는 NaN으로 남을 수 있습니다.
renamed_df["grade_score"] = renamed_df["grade"].map(grade_map)

# replace(): 특정 값을 다른 값으로 치환.
# map()은 변환표에 없는 값이 나오면 “어떻게 바꿀지 모르겠는데?” 하면서 NaN으로 만들 수 있다.
# replace()는 변환표에 없는 값은 “내가 바꾸라고 한 값이 아니네” 하고 그대로 둔다.
# 여기서는 영어 등급명을 한글 라벨로 변경
renamed_df["grade_label"] = renamed_df["grade"].replace({
    "basic": "일반",
    "vip": "우수",
    "vvip": "최우수",
})


# to_datetime(): 문자열로 저장된 날짜를 날짜/시간 타입으로 변환.
# 날짜 타입으로 바꾸면 월별 집계, 기간 계산, 정렬을 더 안정적으로 할 수 있음
renamed_df["join_date"] = pd.to_datetime(renamed_df["join_date"])

# astype(): 자료형을 변환
# age에는 결측치가 있으므로 바로 int로 바꾸면 오류 발생함
# 그래서 평균값으로 결측치를 채우고, 반올림한 뒤 정수형으로 변환합니다.
renamed_df["age_fill"] = renamed_df["age"].fillna(renamed_df["age"].mean()).round().astype(int)

# 변환 전후를 비교하기 위해 원본 등급/나이 컬럼과 새 컬럼을 함께 출력합니다.
display(renamed_df[["id", "name", "grade", "grade_score", "grade_label", "join_date", "age", "age_fill"]])

# info()로 join_date가 datetime 타입이 되었는지, age_fill이 int 타입인지 확인합니다.
renamed_df.info()


### 03-3. 결측치 처리 보강: `notna`, 평균대치, 단순확률대치
결측치 처리는 무조건 삭제하는 작업이 아님

- 삭제: 행 수가 충분하고, 결측치가 분석에 큰 영향을 주지 않을 때
- 평균/중앙값 대치: 수치형 변수의 대표값으로 채울 때
- 최빈값 대치: 범주형 변수에서 가장 자주 등장한 값으로 채울 때
- 단순확률대치: 실제 관측된 값 중 하나를 무작위로 뽑아 채울 때

`isna()`가 결측치인지 확인한다면, `notna()`는 **결측치가 아닌 정상 값인지** 확인한다.


In [ ]:
# customer_work_df를 복사
missing_df = customer_work_df.copy()

print("컬럼별 결측치 수")
# isna()는 결측치이면 True, 아니면 False를 반환합니다.
# sum()을 붙이면 True가 1로 계산되어 컬럼별 결측치 개수를 구할 수 있습니다.
print(missing_df.isna().sum())

print("\n컬럼별 정상값 수")
# notna()는 isna()의 반대입니다.
# 결측치가 아닌 정상값의 개수를 확인할 때 사용합니다.
print(missing_df.notna().sum())

In [ ]:
# 평균대치: 나이 결측치를 평균 나이로 채움.
# 장점: 간단하고 빠릅니다.
# 단점: 데이터의 퍼짐이 줄어들 수 있고, 실제 개별 특성을 반영하지 못할 수 있음
mean_age = missing_df["age"].mean()
missing_df["age_mean_fill"] = missing_df["age"].fillna(mean_age).round(1)
display(missing_df)

# 단순확률대치: 이미 관측된 income 값 중 하나를 무작위로 뽑아 결측치를 채움.
# 평균 하나로 모두 채우는 대신 실제 데이터에 존재하던 값 중에서 뽑는 방식.
# random generator의 seed를 고정하면 매번 같은 결과가 나와 수업 시연이 안정적입니다.
# rng = np.random.default_rng(42)
rng = np.random.default_rng()

# 결측치가 아닌 income 값만 후보로 사용합니다.
# Series.dropna(): 결측치를 제거한 새로운 Series 반환
# Series.to_numpy: Series를 ndarray로 변환
known_income = missing_df["income"].dropna().to_numpy()
print("known_income:", known_income, type(known_income))

# income이 결측치인 행의 index를 찾습니다.
missing_income_idx = missing_df[missing_df["income"].isna()].index
print("missing_income_idx:", missing_income_idx) # 2

# 원본 income 컬럼을 복사해 새 컬럼을 만든 뒤, 결측 위치만 대체합니다.
missing_df["income_random_fill"] = missing_df["income"]

# np.random.choice(ndarray)
# - 배열에서 무작위로 값을 추출합니다.
missing_df.loc[missing_income_idx, "income_random_fill"] = rng.choice(
    known_income,
    size=len(missing_income_idx), # 1
    replace=True, # 복원 추출
)

# 원본 값과 대치 결과를 나란히 비교합니다.
display(missing_df[[
    "customer_id", "age", "age_mean_fill",
    "income", "income_random_fill"
]])


### 03-4. 이상값 처리: IQR 기준과 표준편차 기준
-
- 이상값: 단순히 “큰 값”이 아니라, **전체 데이터 흐름에서 유난히 튀는 값**.
- 이상값을 처리할 때는 바로 삭제하기보다 먼저 원인을 확인해야한다.
---
- 입력 오류인지
- 실제로 매우 특이한 고객인지
- 분석 목적상 유지해야 하는 값인지
- 모델 학습에 과도한 영향을 줄 값인지


In [ ]:
import matplotlib.pyplot as plt

# 이상값 설명용 예제 데이터입니다.
# 대부분의 점수는 50~60 근처에 있지만, 500은 다른 값들과 매우 멀리 떨어져 있습니다.
outlier_df = pd.DataFrame({
    "score": [50, 52, 53, 54, 55, 56, 57, 58, 59, 60, 500]
})

# IQR(Interquartile Range) 방식은 사분위수를 이용해 이상값 후보를 찾습니다.
# IQR = 3사분위수(Q3) - 1사분위수(Q1)
# q1은 하위 25% 지점, q3는 상위 25% 지점입니다.
q1 = outlier_df["score"].quantile(0.25)
q3 = outlier_df["score"].quantile(0.75)

# IQR은 가운데 50% 데이터가 퍼져 있는 범위입니다.
iqr = q3 - q1

# 일반적으로 Q1 - 1.5*IQR보다 작거나 Q3 + 1.5*IQR보다 크면 이상값 후보로 봅니다.
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# 각 행의 score가 IQR 기준 경계 밖에 있는지 True/False로 표시합니다.
outlier_df["is_outlier_iqr"] = (
    (outlier_df["score"] < lower_bound) |
    (outlier_df["score"] > upper_bound)
)
print('outlier_df["is_outlier_iqr"]:', outlier_df["is_outlier_iqr"]) # 10번 인덱스만 True

# 표준편차 기준 이상값도 함께 계산해 비교합니다.
# ddof=0은 현재 데이터를 모집단처럼 보고 표준편차를 계산하겠다는 의미입니다.
score_mean = outlier_df["score"].mean() # 점수 평균
score_std = outlier_df["score"].std(ddof=0) # 점수 표준편차
outlier_df["z_score"] = (outlier_df["score"] - score_mean) / score_std
print('outlier_df["z_score"]: ', outlier_df["z_score"])

# z_score의 절댓값이 3 이상이면 평균에서 표준편차 3개 이상 떨어진 값으로 봄
outlier_df["is_outlier_std3"] = outlier_df["z_score"].abs() >= 3

# 값을 삭제하지 않고 경계값으로 제한하는 방법
# clip()은 lower_bound보다 작은 값은 lower_bound로, upper_bound보다 큰 값은 upper_bound로 바꿉니다.
outlier_df["score_capped"] = outlier_df["score"].clip(lower_bound, upper_bound)

display(outlier_df)
print("IQR 기준 하한:", lower_bound)
print("IQR 기준 상한:", upper_bound)

# boxplot은 중앙값, 사분위수, 이상값 후보를 한 번에 보여주는 그래프입니다.
plt.figure(figsize=(6, 3))
plt.boxplot(outlier_df["score"], orientation="horizontal")
plt.title("Score outlier check")
plt.xlabel("score")
plt.show()


### 03-5. 데이터 정규화: Z-score, Min-Max
- 정규화는 값의 범위를 맞추는 작업
    - 예를 들어 `income`은 천 단위이고, `purchase_count`는 한 자리 수라면 두 컬럼의 숫자 크기가 너무 다르다는 문제 발생.
    - 거리 기반 알고리즘이나 경사하강법 기반 모델에서는 값의 크기 차이가 학습에 영향을 줄 수 있습니다.
        - (참고) 경사하강법: 기반 모델은 예측 오차를 줄이는 방향으로 가중치를 조금씩 조정하면서 학습하는 모델.
        - 데이터 값의 크기 차이가 크면 학습이 불안정해질 수 있어, 보통 정규화나 표준화를 함께 사용함.

- Z-score 표준화: 평균을 0, 표준편차를 1에 가깝게 변환
- Min-Max 정규화: 값을 0부터 1 사이로 변환


In [ ]:
# 단위와 숫자 크기가 서로 다른 score, income, purchase_count
scale_df = pd.DataFrame({
    "score": [45, 55, 60, 70, 95],
    "income": [2500, 3200, 4000, 5200, 9000],
    "purchase_count": [1, 2, 2, 4, 8],
})

# Z-score 표준화: 값이 평균에서 표준편차 몇 개만큼 떨어져 있는지 나타냅니다.
# 공식: z = (값 - 평균) / 표준편차
# 변환 후에는 평
score_std = scale_df["score"].std(ddof=0) # 표준편차
scale_df["score_z"] = (scale_df["score"] - score_mean) / score_std
print('scale_df["score_z"]:', scale_df["score_z"])

# Min-Max 정규화: 최소값은 0, 최대값은 1이 되도록 변환합니다.
# 공식: (값 - 최소값) / (최대값 - 최소값)
# 값의 범위를 0~1 사이로 맞추고 싶을 때 자주 사용합니다.
income_min = scale_df["income"].min()
income_max = scale_df["income"].max()
scale_df["income_minmax"] = (scale_df["income"] - income_min) / (income_max - income_min)
print('scale_df["income_minmax"]:', scale_df["income_minmax"])

# round(3)은 표를 보기 좋게 소수점 3자리로 제한합니다.
display(scale_df.round(3))
print("정규화/표준화 결과에서는 “원래 값이 얼마냐”보다 “평균에서 얼마나 떨어졌는지”, “최소~최대 사이 어디쯤 있는지”를 본다.")


### 03-6. 중심극한정리 맛보기
- “원래 데이터가 꼭 정규분포가 아니어도, 표본을 여러 번 뽑아 평균을 구하면 그 평균들의 분포는 정규분포에 가까워진다”는 내용


In [ ]:
rng = np.random.default_rng(42)

# 주사위를 1000번 던진 결과
# 1~6 사이의 정수가 비슷한 빈도로 나오므로 원래 데이터는 균등분포에 가깝다.
single_rolls = rng.integers(1, 7, size=1000)

# 표본평균을 1000개 생성
# 한 번에 주사위를 30번 던져 평균을 구하고, 이 과정을 1000번 반복
# 중심극한정리의 핵심은 "개별 값"이 아니라 "표본평균들의 분포"를 보는 것!
sample_means = np.array([
    rng.integers(1, 7, size=30).mean()
    for _ in range(1000)
])

# 1행 2열 그래프를 만들어 왼쪽에는 개별 주사위 값, 오른쪽에는 표본평균 분포를 비교
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# bins를 0.5 단위로 밀어 주면 1, 2, 3, 4, 5, 6이 막대 중앙에 오도록 그릴 수 있음
axes[0].hist(single_rolls, bins=np.arange(1, 8) - 0.5, edgecolor="white")
axes[0].set_title("Single dice rolls")
axes[0].set_xlabel("dice value")
axes[0].set_ylabel("count")

# 표본평균은 1~6 전체에 고르게 퍼지지 않고, 평균값인 3.5 근처에 많이 모임
axes[1].hist(sample_means, bins=30, edgecolor="white")
axes[1].set_title("Means of 30 dice rolls")
axes[1].set_xlabel("sample mean")
axes[1].set_ylabel("count")

# 제목과 축 라벨이 겹치지 않도록 여백을 자동 조정
plt.tight_layout()
plt.show()

print("주사위 값 자체는 1~6이 비슷하게 나오지만, 30번씩 뽑은 평균은 가운데 값 주변에 많이 모여있음")


### 03-7. 다중대치 맛보기: `IterativeImputer`
- 결측값을 단순 평균으로 채우는 대신, 다른 변수들과의 관계를 이용해 결측값을 추정하는 방식
- 예를 들어 구매금액, 소득, 나이가 서로 어느 정도 관련이 있다면, 이 관계를 활용해 비어 있는 값을 더 그럴듯하게 채울 수 있음.
- 머신러닝의 `scikit-learn(사이킷런)`과 함께 더 자세히 다루면 좋음


In [ ]:
!pip install scikit-learn

In [ ]:
# 다중대치 예제에서는 수치형 컬럼만 사용
# IterativeImputer는 여러 컬럼의 관계를 이용해 결측값을 추정하므로 문자열 컬럼은 제외
impute_input_df = customer_work_df[["purchase1", "purchase2", "income", "age"]]
display(impute_input_df)

# IterativeImputer는 scikit-learn에서 아직 experimental 기능으로 제공된다.
# 그래서 enable_iterative_imputer를 먼저 import해야 사용할 수 있음
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# random_state는 재현 가능한 결과를 위해 고정
# max_iter는 결측값 추정을 반복할 최대 횟수
imputer = IterativeImputer(random_state=42, max_iter=10)

# fit_transform(): 데이터의 패턴을 학습하고 결측치를 채운 배열을 반환
# 반환값은 NumPy 배열이므로 다시 DataFrame으로 바꾸면 비교하기 좋음
imputed_array = imputer.fit_transform(impute_input_df)

imputed_df = pd.DataFrame(
    imputed_array,
    columns=["purchase1_fill", "purchase2_fill", "income_fill", "age_fill"],
).round(1)

display(imputed_df)

